In [ ]:

import os
import pandas as pd

from datetime import datetime, timezone
from dotenv import load_dotenv
from influxdb_client import InfluxDBClient

# config

load_dotenv()

client = InfluxDBClient(
    url=os.getenv("INFLUXDB_URL"),
    token=os.getenv("INFLUXDB_TOKEN"),
    org=os.getenv("INFLUXDB_ORG")
)

bucket = os.getenv("INFLUXDB_BUCKET")

output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)

start_date = datetime(2026, 4, 6, tzinfo=timezone.utc)
end_date = datetime(2026, 4, 13, tzinfo=timezone.utc)


# check connection

if not client.ping():
    client.close()
    raise ConnectionError("Geen verbinding met InfluxDB!")

print("Verbinding met InfluxDB succesvol!")


# Get data

query = f'''
from(bucket: "{bucket}")
    |> range(
        start: {start_date.isoformat()},
        stop: {end_date.isoformat()}
    )
    |> pivot(
        rowKey: ["_time"],
        columnKey: ["_field"],
        valueColumn: "_value"
    )
'''

try:

    df = client.query_api().query_data_frame(query)

    # combine results if neccesary
    if isinstance(df, list):
        df = pd.concat(df, ignore_index=True)

    # save data

    if not df.empty:

        filename = "diopsis_data.csv"
        filepath = os.path.join(output_dir, filename)

        df.to_csv(filepath, index=False)

        print(f"Export success!")
        print(f"Records: {len(df):,}")
        print(f"File saved: {filepath}")

    else:
        print("Geen data gevonden in deze periode.")

except Exception as error:
    print(f"Fout tijdens export: {error}")

finally:
    client.close()


True